In [52]:
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))


# Lab | Natural Language Processing
### SMS: SPAM or HAM

### Let's prepare the environment

In [53]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer

- Read Data for the Fraudulent Email Kaggle Challenge
- Reduce the training set to speead up development. 

In [54]:
import os

print(os.getcwd())

c:\Users\aysea\AI_Engineering_Bootcamp\lab-natural-language-processing\your-code


In [55]:
import os

print(os.listdir())

['main.ipynb']


In [56]:
import os

print("Current folder:")
print(os.getcwd())

print("\nFiles:")
print(os.listdir())

Current folder:
c:\Users\aysea\AI_Engineering_Bootcamp\lab-natural-language-processing\your-code

Files:
['main.ipynb']


In [57]:
import os

for root, dirs, files in os.walk(".."):
    for file in files:
        if file.endswith(".csv"):
            print(os.path.join(root, file))

..\data\kg_test.csv
..\data\kg_train.csv


In [58]:
import pandas as pd

train_df = pd.read_csv("../data/kg_train.csv")

train_df.head()

,text,label
0,"DEAR SIR, STRICTLY A PRIVATE BUSINESS PROPOSAL...",1
1,Will do.,0
2,Nora--Cheryl has emailed dozens of memos about...,0
3,Dear Sir=2FMadam=2C I know that this proposal ...,1
4,fyi,0


In [59]:
print(train_df.shape)

(5964, 2)


In [60]:
print(train_df.columns)

Index(['text', 'label'], dtype='str')


In [61]:
test_df = pd.read_csv("../data/kg_test.csv")

test_df.head()

,text
0,usiness is for the fact that the deceased man ...
1,They are happy to adjust to the afternoon. I a...
2,Lael Brainard was confirmed 78-19 this afterno...
3,H <hrod17@clintonemail.com>Friday March 26 201...
4,"n;""> Dear Good Friend,<br><br><br>I am happy t..."


In [62]:
# reducing the dataset size for faster training
small_train = train_df.sample(
    n=5000,
    random_state=42
)


In [63]:
print(small_train.shape)

(5000, 2)


In [64]:
train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5964 entries, 0 to 5963
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   text    5964 non-null   str  
 1   label   5964 non-null   int64
dtypes: int64(1), str(1)
memory usage: 93.3 KB


In [65]:
train_df.columns

Index(['text', 'label'], dtype='str')

In [66]:
train_df.head()

,text,label
0,"DEAR SIR, STRICTLY A PRIVATE BUSINESS PROPOSAL...",1
1,Will do.,0
2,Nora--Cheryl has emailed dozens of memos about...,0
3,Dear Sir=2FMadam=2C I know that this proposal ...,1
4,fyi,0


In [67]:
train_df.isnull().sum()

text     0
label    0
dtype: int64

In [68]:
print(train_df.columns)

Index(['text', 'label'], dtype='str')


### Let's divide the training and test set into two partitions

In [69]:
from sklearn.model_selection import train_test_split

data_train, data_val = train_test_split(
    data,
    test_size=0.2,
    random_state=42,
    stratify=data['label']
)
print("Train shape:", data_train.shape)
print("Validation shape:", data_val.shape)


NameError: name 'data' is not defined

In [ ]:
train_df = pd.read_csv("../data/kg_train.csv")

In [ ]:
from sklearn.model_selection import train_test_split

data_train, data_val = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42
)

In [ ]:
stratify=train_df['label']

In [ ]:
print(train_df.columns)

Index(['text', 'label'], dtype='str')


In [ ]:
import pandas as pd

train_df = pd.read_csv("../data/kg_train.csv")

In [ ]:
print(train_df.head())

                                                text  label
0  DEAR SIR, STRICTLY A PRIVATE BUSINESS PROPOSAL...      1
1                                           Will do.      0
2  Nora--Cheryl has emailed dozens of memos about...      0
3  Dear Sir=2FMadam=2C I know that this proposal ...      1
4                                                fyi      0


## Data Preprocessing

In [ ]:
import string
from nltk.corpus import stopwords
print(string.punctuation)
print(stopwords.words("english")[100:110])
from nltk.stem.snowball import SnowballStemmer
snowball = SnowballStemmer('english')

## Now, we have to clean the html code removing words

- First we remove inline JavaScript/CSS
- Then we remove html comments. This has to be done before removing regular tags since comments can contain '>' characters
- Next we can remove the remaining tags

In [ ]:
import re
import warnings
from bs4 import BeautifulSoup, MarkupResemblesLocatorWarning

warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)

def clean_html_and_text(text):
    if text is None:
        return ""
    text = str(text)
    text = re.sub(r'(?is)<script.*?>.*?</script>', ' ', text)
    text = re.sub(r'(?is)<style.*?>.*?</style>', ' ', text)
    text = re.sub(r'(?is)<!--.*?-->', ' ', text)
    text = re.sub(r'(?is)<[^>]+>', ' ', text)
    text = BeautifulSoup(text, "html.parser").get_text(" ", strip=True)
    text = text.replace("\n", " " ).replace("\r", " ")
    text = text.lower()
    text = re.sub(r"^b[\"']", "", text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\b[a-z]\b', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

clean_html_and_text("<b>Hello</b> 123 world!")


- Remove all the special characters
    
- Remove numbers
    
- Remove all single characters
 
- Remove single characters from the start

- Substitute multiple spaces with single space

- Remove prefixed 'b'

- Convert to Lowercase

In [ ]:
for df in (data_train, data_val):
    df['preprocessed_text'] = df['text'].apply(clean_html_and_text)

for df in (data_train, data_val):
    df['preprocessed_text'] = df['preprocessed_text'].apply(lambda x: re.sub(r'[^a-z\s]', ' ', x))
    df['preprocessed_text'] = df['preprocessed_text'].apply(lambda x: re.sub(r'\b[a-z]\b', ' ', x))
    df['preprocessed_text'] = df['preprocessed_text'].apply(lambda x: re.sub(r'\s+', ' ', x).strip())

print(data_train[['text', 'preprocessed_text']].head())


## Now let's work on removing stopwords
Remove the stopwords.

In [ ]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    tokens = [word for word in text.split() if word not in stop_words and len(word) > 1]
    return ' '.join(tokens)

for df in (data_train, data_val):
    df['preprocessed_text'] = df['preprocessed_text'].apply(remove_stopwords)

print(data_train['preprocessed_text'].head())


## Tame Your Text with Lemmatization
Break sentences into words, then use lemmatization to reduce them to their base form (e.g., "running" becomes "run"). See how this creates cleaner data for analysis!

In [ ]:
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

def lemmatize_text(text):
    return ' '.join(lemmatizer.lemmatize(token) for token in text.split())

for df in (data_train, data_val):
    df['preprocessed_text'] = df['preprocessed_text'].apply(lemmatize_text)

print(data_train['preprocessed_text'].head())


## Bag Of Words
Let's get the 10 top words in ham and spam messages (**EXPLORATORY DATA ANALYSIS**)

In [ ]:
from collections import Counter

ham_words = Counter()
spam_words = Counter()

for text, label in zip(data_train['preprocessed_text'], data_train['label']):
    words = text.split()
    if label == 0:
        ham_words.update(words)
    else:
        spam_words.update(words)

print('Top 10 ham words:', ham_words.most_common(10))
print('Top 10 spam words:', spam_words.most_common(10))

ham_df = pd.DataFrame(ham_words.most_common(10), columns=['word', 'count'])
spam_df = pd.DataFrame(spam_words.most_common(10), columns=['word', 'count'])
print(ham_df)
print(spam_df)


## Extra features

In [ ]:
# We add to the original dataframe two additional indicators (money symbols and suspicious words).
money_simbol_list = "|".join(["euro","dollar","pound","€",r"\$"])
suspicious_words = "|".join(["free","cheap","sex","money","account","bank","fund","transfer","transaction","win","deposit","password"])

data_train['money_mark'] = data_train['preprocessed_text'].str.contains(money_simbol_list)*1
data_train['suspicious_words'] = data_train['preprocessed_text'].str.contains(suspicious_words)*1
data_train['text_len'] = data_train['preprocessed_text'].apply(lambda x: len(x)) 

data_val['money_mark'] = data_val['preprocessed_text'].str.contains(money_simbol_list)*1
data_val['suspicious_words'] = data_val['preprocessed_text'].str.contains(suspicious_words)*1
data_val['text_len'] = data_val['preprocessed_text'].apply(lambda x: len(x)) 

data_train.head()

## How would work the Bag of Words with Count Vectorizer concept?

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

count_vectorizer = CountVectorizer()
X_train_bow = count_vectorizer.fit_transform(data_train['preprocessed_text'])
X_val_bow = count_vectorizer.transform(data_val['preprocessed_text'])

model_bow = MultinomialNB()
model_bow.fit(X_train_bow, data_train['label'])

pred_bow = model_bow.predict(X_val_bow)
print('Bag-of-words shape (train):', X_train_bow.shape)
print('Bag-of-words shape (validation):', X_val_bow.shape)
print('Bag-of-words accuracy:', accuracy_score(data_val['label'], pred_bow))
print(classification_report(data_val['label'], pred_bow))


## TF-IDF

- Load the vectorizer

- Vectorize all dataset

- print the shape of the vetorized dataset

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf_vectorizer.fit_transform(data_train['preprocessed_text'])
X_val_tfidf = tfidf_vectorizer.transform(data_val['preprocessed_text'])

model_tfidf = MultinomialNB()
model_tfidf.fit(X_train_tfidf, data_train['label'])

pred_tfidf = model_tfidf.predict(X_val_tfidf)
print('TF-IDF shape (train):', X_train_tfidf.shape)
print('TF-IDF shape (validation):', X_val_tfidf.shape)
print('TF-IDF accuracy:', accuracy_score(data_val['label'], pred_tfidf))
print(classification_report(data_val['label'], pred_tfidf))


## And the Train a Classifier?

In [ ]:
models = {
    'Bag of Words': model_bow,
    'TF-IDF': model_tfidf,
}

for name, model in models.items():
    if name == 'Bag of Words':
        X_val = X_val_bow
    else:
        X_val = X_val_tfidf
    y_pred = model.predict(X_val)
    acc = accuracy_score(data_val['label'], y_pred)
    print(f'{name} validation accuracy: {acc:.4f}')

print('Best approach in this experiment: TF-IDF only')


### Extra Task - Implement a SPAM/HAM classifier

https://www.kaggle.com/t/b384e34013d54d238490103bc3c360ce

The classifier can not be changed!!! It must be the MultinimialNB with default parameters!

Your task is to **find the most relevant features**.

For example, you can test the following options and check which of them performs better:
- Using "Bag of Words" only
- Using "TF-IDF" only
- Bag of Words + extra flags (money_mark, suspicious_words, text_len)
- TF-IDF + extra flags


You can work with teams of two persons (recommended).

In [ ]:
import numpy as np

for df in (data_train, data_val):
    df['money_mark'] = df['preprocessed_text'].str.lower().apply(
        lambda text: any(term in text for term in ['euro', 'dollar', 'pound', '$'])
    ).astype(int)
    df['suspicious_words'] = df['preprocessed_text'].str.lower().apply(
        lambda text: any(term in text for term in [
            'free', 'cheap', 'sex', 'money', 'account', 'bank', 'fund',
            'transfer', 'transaction', 'win', 'deposit', 'password'
        ])
    ).astype(int)
    df['text_len'] = df['preprocessed_text'].apply(lambda x: len(x))

extra_cols = ['money_mark', 'suspicious_words', 'text_len']

X_train_bow_extra = np.hstack([X_train_bow.toarray(), data_train[extra_cols].values])
X_val_bow_extra = np.hstack([X_val_bow.toarray(), data_val[extra_cols].values])
model_bow_extra = MultinomialNB().fit(X_train_bow_extra, data_train['label'])
print('Bag of Words + extra flags accuracy:', accuracy_score(data_val['label'], model_bow_extra.predict(X_val_bow_extra)))

X_train_tfidf_extra = np.hstack([X_train_tfidf.toarray(), data_train[extra_cols].values])
X_val_tfidf_extra = np.hstack([X_val_tfidf.toarray(), data_val[extra_cols].values])
model_tfidf_extra = MultinomialNB().fit(X_train_tfidf_extra, data_train['label'])
print('TF-IDF + extra flags accuracy:', accuracy_score(data_val['label'], model_tfidf_extra.predict(X_val_tfidf_extra)))

feature_names = np.array(tfidf_vectorizer.get_feature_names_out())
class_diff = model_tfidf.feature_log_prob_[1] - model_tfidf.feature_log_prob_[0]
spam_feature_indexes = np.argsort(class_diff)[::-1][:20]
print('Top spam features:', feature_names[spam_feature_indexes].tolist())
print('Most relevant spam words are the ones with the largest positive difference between spam and ham log probabilities.')
